<a href="https://colab.research.google.com/github/saim9211/Machine-Learning-Stuff/blob/main/random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# performing the random forest classifier


In [1]:
import kagglehub
path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction")

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


In [2]:
import pandas as pd
import os

# Assuming the CSV file is named 'heart.csv' within the downloaded path
data_file_path = os.path.join(path, 'heart.csv')
df = pd.read_csv(data_file_path)

print("DataFrame head:")
print(df.head())

DataFrame head:
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [3]:
df.sample(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
15,54,F,ATA,120,273,0,Normal,150,N,1.5,Flat,0
608,62,M,ASY,158,170,0,ST,138,Y,0.0,Flat,1
206,35,M,ATA,120,308,0,LVH,180,N,0.0,Up,0
753,34,F,ATA,118,210,0,Normal,192,N,0.7,Up,0
238,48,M,ASY,160,355,0,Normal,99,Y,2.0,Flat,1


In [5]:
df.isnull().sum()

,0
Age,0
Sex,0
ChestPainType,0
RestingBP,0
Cholesterol,0
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [12]:
import pandas as pd

# Identify categorical columns (object dtype)
categorical_cols = df.select_dtypes(include='object').columns

# Apply one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("DataFrame after one-hot encoding:")
print(df_encoded.head())
print("\nDataFrame info after encoding:")
df_encoded.info()

# Re-split the data after encoding
x = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\nx_train shape after encoding and splitting:", x_train.shape)
print("x_test shape after encoding and splitting:", x_test.shape)

DataFrame after one-hot encoding:
   Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak  HeartDisease  \
0   40        140          289          0    172      0.0             0   
1   49        160          180          0    156      1.0             1   
2   37        130          283          0     98      0.0             0   
3   48        138          214          0    108      1.5             1   
4   54        150          195          0    122      0.0             0   

   Sex_M  ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  \
0   True               True              False             False   
1  False              False               True             False   
2   True               True              False             False   
3  False              False              False             False   
4   True              False               True             False   

   RestingECG_Normal  RestingECG_ST  ExerciseAngina_Y  ST_Slope_Flat  \
0               True          Fals

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [7]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(['HeartDisease'],axis=1),df['HeartDisease'],test_size=0.2,random_state=42)


In [14]:
rc = RandomForestClassifier(n_estimators=100, random_state=42)
rc.fit(x_train, y_train)

RandomForestClassifier(random_state=42)

In [15]:
y_pred = rc.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.8750


In [16]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [4, 6, 8, None],
    'criterion': ['gini', 'entropy']
}

# Create a new Random Forest Classifier instance
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search.fit(x_train, y_train)

# Print the best parameters and best score
print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Evaluate the best estimator on the test set
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(x_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Accuracy on the test set with tuned parameters: {accuracy_tuned:.4f}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best parameters found: {'criterion': 'entropy', 'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
Best cross-validation accuracy: 0.8774
Accuracy on the test set with tuned parameters: 0.8641


In [17]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define a broader parameter distribution for Randomized Search
param_dist = {
    'n_estimators': randint(100, 500), # Number of trees in the forest
    'max_features': ['sqrt', 'log2', None], # Number of features to consider at every split
    'max_depth': randint(4, 15), # Maximum number of levels in tree
    'min_samples_split': randint(2, 20), # Minimum number of samples required to split a node
    'min_samples_leaf': randint(1, 20), # Minimum number of samples required at each leaf node
    'criterion': ['gini', 'entropy'] # Function to measure the quality of a split
}

# Create a new Random Forest Classifier instance
rf_random = RandomForestClassifier(random_state=42)

# Initialize RandomizedSearchCV
# n_iter specifies the number of parameter settings that are sampled
random_search = RandomizedSearchCV(estimator=rf_random, param_distributions=param_dist,
                                   n_iter=100, cv=5, scoring='accuracy', random_state=42,
                                   n_jobs=-1, verbose=1)

# Fit RandomizedSearchCV to the training data
print("\nStarting RandomizedSearchCV...")
random_search.fit(x_train, y_train)

# Print the best parameters and best score
print(f"\nBest parameters found by RandomizedSearchCV: {random_search.best_params_}")
print(f"Best cross-validation accuracy by RandomizedSearchCV: {random_search.best_score_:.4f}")

# Evaluate the best estimator on the test set
best_rf_random = random_search.best_estimator_
y_pred_random_tuned = best_rf_random.predict(x_test)
accuracy_random_tuned = accuracy_score(y_test, y_pred_random_tuned)
print(f"Accuracy on the test set with RandomizedSearchCV tuned parameters: {accuracy_random_tuned:.4f}")


Starting RandomizedSearchCV...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best parameters found by RandomizedSearchCV: {'criterion': 'entropy', 'max_depth': 12, 'max_features': 'log2', 'min_samples_leaf': 3, 'min_samples_split': 9, 'n_estimators': 273}
Best cross-validation accuracy by RandomizedSearchCV: 0.8746
Accuracy on the test set with RandomizedSearchCV tuned parameters: 0.8641
